# q02：ベースラインVADの実装とパラメータ決定

基礎レベル2では，短時間エネルギーに基づくベースラインVADを実装し，trainデータに対するmacro-F1で $\theta$ と $L_h$ を決定する．

評価条件は10 msフレーム，A1からA6をtrainとする．

In [ ]:
import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import json



from pathlib import Path

ROOT = Path("./data")
INPUT_DIR = ROOT / "inputs"
ANNOTATION_DIR = ROOT / "annotations"
RESULT_DIR = ROOT / "results"

ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_IDS = [f"A{i}" for i in range(1, 7)]
TEST_IDS = [f"A{i}" for i in range(7, 10)]
ALL_IDS = TRAIN_IDS + TEST_IDS

FRAME_SEC = 0.010
TARGET_FS = 16000

def wav_path(utt_id: str) -> Path:
    return INPUT_DIR / f"{utt_id}.wav"

def interval_path(utt_id: str) -> Path:
    return ANNOTATION_DIR / f"{utt_id}_intervals.csv"

def label_path(utt_id: str) -> Path:
    return ANNOTATION_DIR / f"{utt_id}_labels.csv"



import numpy as np
import pandas as pd
import soundfile as sf

EPS = 1e-12

def load_audio(path, target_fs: int = 16000):
    """WAVを読み込み，モノラルfloat32波形として返す．"""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"音声ファイルが見つかりません: {path}")
    x, fs = sf.read(path, dtype="float32")
    if x.ndim == 2:
        x = x.mean(axis=1)
    if fs != target_fs:
        raise ValueError(f"{path} のサンプリング周波数が {fs} Hz です．{target_fs} Hzで収録した音声を使用してください．")
    return x.astype(np.float32), fs

def frame_signal(x: np.ndarray, fs: int, frame_sec: float = 0.010):
    """10 msごとに非重複フレーム化する．不足分は0で埋める．"""
    frame_len = int(round(frame_sec * fs))
    n_frames = int(np.ceil(len(x) / frame_len))
    pad_len = n_frames * frame_len - len(x)
    if pad_len > 0:
        x = np.pad(x, (0, pad_len))
    frames = x.reshape(n_frames, frame_len)
    times = np.arange(n_frames) * frame_sec
    return frames, times

def hann_window(n: int):
    return np.hanning(n).astype(np.float32)

def short_time_log_energy(x: np.ndarray, fs: int, frame_sec: float = 0.010, use_hann: bool = True):
    frames, times = frame_signal(x, fs, frame_sec)
    if use_hann:
        frames = frames * hann_window(frames.shape[1])[None, :]
    energy = np.mean(frames ** 2, axis=1)
    log_energy = 10.0 * np.log10(EPS + energy)
    return log_energy, times

def relative_energy(log_energy: np.ndarray, percentile: float = 20.0):
    noise_floor = np.percentile(log_energy, percentile)
    score = log_energy - noise_floor
    return score, noise_floor

def apply_hangover(pred: np.ndarray, hangover_frames: int):
    """1から0へ切り替わった直後の指定フレーム数を1にする．"""
    pred = np.asarray(pred, dtype=np.int64).copy()
    if hangover_frames <= 0:
        return pred
    out = pred.copy()
    n = len(pred)
    for m in range(n - 1):
        if pred[m] == 1 and pred[m + 1] == 0:
            end = min(n, m + 1 + hangover_frames)
            out[m + 1:end] = 1
    return out

def labels_from_intervals(intervals, n_frames: int, frame_sec: float = 0.010):
    """[(start_sec, end_sec), ...]を10 msフレームラベルへ変換する．"""
    labels = np.zeros(n_frames, dtype=np.int64)
    for start_sec, end_sec in intervals:
        start = max(0, int(np.floor(start_sec / frame_sec)))
        end = min(n_frames, int(np.ceil(end_sec / frame_sec)))
        labels[start:end] = 1
    return labels

def save_labels_from_intervals(utt_id: str, intervals, duration_sec: float):
    """区間アノテーションCSVとフレームラベルCSVを保存する．"""
    n_frames = int(np.ceil(duration_sec / FRAME_SEC))
    interval_df = pd.DataFrame(intervals, columns=["start_sec", "end_sec"])
    interval_df["label"] = 1
    interval_df.to_csv(interval_path(utt_id), index=False)

    labels = labels_from_intervals(intervals, n_frames, FRAME_SEC)
    label_df = pd.DataFrame({
        "frame": np.arange(n_frames),
        "start_sec": np.arange(n_frames) * FRAME_SEC,
        "end_sec": (np.arange(n_frames) + 1) * FRAME_SEC,
        "label": labels
    })
    label_df.to_csv(label_path(utt_id), index=False)
    return interval_df, label_df

def load_frame_labels(utt_id: str, n_frames: int):
    """q01で保存したフレームラベルを読み込む．長さが合わない場合は切り詰めまたは0埋めする．"""
    path = label_path(utt_id)
    if not path.exists():
        raise FileNotFoundError(f"アノテーションが見つかりません: {path}")
    labels = pd.read_csv(path)["label"].to_numpy(dtype=np.int64)
    if len(labels) < n_frames:
        labels = np.pad(labels, (0, n_frames - len(labels)))
    elif len(labels) > n_frames:
        labels = labels[:n_frames]
    return labels

def confusion_binary(y_true: np.ndarray, y_pred: np.ndarray, positive_label: int = 1):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tp = int(np.sum((y_true == positive_label) & (y_pred == positive_label)))
    fp = int(np.sum((y_true != positive_label) & (y_pred == positive_label)))
    fn = int(np.sum((y_true == positive_label) & (y_pred != positive_label)))
    tn = int(np.sum((y_true != positive_label) & (y_pred != positive_label)))
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn}

def f1_from_counts(tp: int, fp: int, fn: int):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return f1, precision, recall

def macro_f1_score(y_true: np.ndarray, y_pred: np.ndarray):
    """音声クラスと非音声クラスのF1平均を返す．"""
    c_s = confusion_binary(y_true, y_pred, positive_label=1)
    f1_s, p_s, r_s = f1_from_counts(c_s["TP"], c_s["FP"], c_s["FN"])

    c_n = confusion_binary(1 - y_true, 1 - y_pred, positive_label=1)
    f1_n, p_n, r_n = f1_from_counts(c_n["TP"], c_n["FP"], c_n["FN"])

    return {
        "macro_f1": (f1_s + f1_n) / 2,
        "f1_speech": f1_s,
        "precision_speech": p_s,
        "recall_speech": r_s,
        "f1_non_speech": f1_n,
        "precision_non_speech": p_n,
        "recall_non_speech": r_n,
        **c_s
    }

def load_dataset(utt_ids):
    """音声，エネルギー，正解ラベルをIDごとに読み込む．"""
    data = {}
    for utt_id in utt_ids:
        x, fs = load_audio(wav_path(utt_id), TARGET_FS)
        E, times = short_time_log_energy(x, fs, FRAME_SEC, use_hann=True)
        y = load_frame_labels(utt_id, len(E))
        data[utt_id] = {"x": x, "fs": fs, "E": E, "times": times, "y": y}
    return data

def concat_labels_and_preds(y_list, pred_list):
    return np.concatenate(y_list), np.concatenate(pred_list)

## 1．trainデータの読み込み

In [ ]:
train_data = load_dataset(TRAIN_IDS)
print("loaded train files:", list(train_data.keys()))

for uid, d in train_data.items():
    print(uid, "frames:", len(d["E"]), "speech frames:", int(d["y"].sum()))

## 2．ベースライン条件の定義

次の3条件を比較する．

1. エネルギーしきい値のみ  
2. 相対エネルギーによるしきい値  
3. 相対エネルギー + hangover

In [ ]:
def predict_energy_only(E: np.ndarray, theta: float):
    return (E >= theta).astype(np.int64)

def predict_relative_energy(E: np.ndarray, theta: float):
    S, noise_floor = relative_energy(E, percentile=20.0)
    return (S >= theta).astype(np.int64)

def predict_relative_energy_hangover(E: np.ndarray, theta: float, Lh: int):
    pred0 = predict_relative_energy(E, theta)
    return apply_hangover(pred0, Lh)

def evaluate_on_dataset(data, predictor):
    y_all = []
    pred_all = []
    for uid, d in data.items():
        pred = predictor(d["E"])
        y_all.append(d["y"])
        pred_all.append(pred)
    y_all, pred_all = concat_labels_and_preds(y_all, pred_all)
    return macro_f1_score(y_all, pred_all)

def make_abs_theta_grid(data):
    values = np.concatenate([d["E"] for d in data.values()])
    lo = np.floor(values.min() * 2) / 2
    hi = np.ceil(values.max() * 2) / 2
    return np.arange(lo, hi + 0.5, 0.5)

def make_rel_theta_grid(data):
    values = []
    for d in data.values():
        S, _ = relative_energy(d["E"], percentile=20.0)
        values.append(S)
    values = np.concatenate(values)
    hi = max(0.0, np.ceil(values.max() * 2) / 2)
    return np.arange(0.0, hi + 0.5, 0.5)

## 3．$\theta$ と $L_h$ の探索

$\theta$ は0.5 dB刻みで探索する．  
hangoverは $L_h \in \{0,3,5,10,20\}$ から選ぶ．  
同じmacro-F1なら，短い $L_h$ を採用する．

In [ ]:
results = []

# 条件1：エネルギーしきい値のみ
for theta in make_abs_theta_grid(train_data):
    metrics = evaluate_on_dataset(
        train_data,
        lambda E, theta=theta: predict_energy_only(E, theta)
    )
    results.append({
        "method": "energy_only",
        "theta": float(theta),
        "Lh": 0,
        **metrics
    })

# 条件2：相対エネルギー
for theta in make_rel_theta_grid(train_data):
    metrics = evaluate_on_dataset(
        train_data,
        lambda E, theta=theta: predict_relative_energy(E, theta)
    )
    results.append({
        "method": "relative_energy",
        "theta": float(theta),
        "Lh": 0,
        **metrics
    })

# 条件3：相対エネルギー + hangover
for theta in make_rel_theta_grid(train_data):
    for Lh in [0, 3, 5, 10, 20]:
        metrics = evaluate_on_dataset(
            train_data,
            lambda E, theta=theta, Lh=Lh: predict_relative_energy_hangover(E, theta, Lh)
        )
        results.append({
            "method": "relative_energy_hangover",
            "theta": float(theta),
            "Lh": int(Lh),
            **metrics
        })

result_df = pd.DataFrame(results)

# methodごとの最良条件を選ぶ
best_by_method = (
    result_df
    .sort_values(["method", "macro_f1", "Lh"], ascending=[True, False, True])
    .groupby("method", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

# 全ベースライン条件の中で最良条件を選ぶ
best_baseline = (
    result_df
    .sort_values(["macro_f1", "Lh"], ascending=[False, True])
    .iloc[0]
    .to_dict()
)

result_df.to_csv(RESULT_DIR / "baseline_train_results.csv", index=False)
best_by_method.to_csv(RESULT_DIR / "baseline_best_by_method.csv", index=False)

with open(RESULT_DIR / "baseline_best_params.json", "w", encoding="utf-8") as f:
    json.dump(best_baseline, f, ensure_ascii=False, indent=2)

display(best_by_method[["method", "theta", "Lh", "macro_f1", "f1_speech", "f1_non_speech", "TP", "FP", "FN", "TN"]])
print("best baseline")
print(json.dumps(best_baseline, ensure_ascii=False, indent=2))

## 4．探索結果の可視化

In [ ]:
plt.figure(figsize=(10, 4))
for method, g in result_df[result_df["Lh"] == 0].groupby("method"):
    if method == "relative_energy_hangover":
        continue
    plt.plot(g["theta"], g["macro_f1"], marker="o", markersize=3, label=method)

plt.xlabel("Threshold [dB]")
plt.ylabel("macro-F1")
plt.title("Baseline threshold search")
plt.grid(True)
plt.legend()
plt.show()

rel_h = result_df[result_df["method"] == "relative_energy_hangover"]
pivot = rel_h.pivot_table(index="theta", columns="Lh", values="macro_f1", aggfunc="max")

plt.figure(figsize=(10, 4))
for Lh in pivot.columns:
    plt.plot(pivot.index, pivot[Lh], marker="o", markersize=3, label=f"Lh={Lh}")
plt.xlabel("Relative energy threshold [dB]")
plt.ylabel("macro-F1")
plt.title("Relative energy + hangover search")
plt.grid(True)
plt.legend()
plt.show()

## 5．A1の判定例

In [ ]:
example_id = "A1"
d = train_data[example_id]

if best_baseline["method"] == "energy_only":
    pred = predict_energy_only(d["E"], best_baseline["theta"])
    score = d["E"]
    score_name = "log energy [dB]"
elif best_baseline["method"] == "relative_energy":
    pred = predict_relative_energy(d["E"], best_baseline["theta"])
    score, _ = relative_energy(d["E"], percentile=20.0)
    score_name = "relative energy [dB]"
else:
    pred = predict_relative_energy_hangover(d["E"], best_baseline["theta"], int(best_baseline["Lh"]))
    score, _ = relative_energy(d["E"], percentile=20.0)
    score_name = "relative energy [dB]"

plt.figure(figsize=(14, 5))
plt.plot(d["times"], score, label=score_name)
plt.plot(d["times"], d["y"] * np.max(score), label="true label", alpha=0.7)
plt.plot(d["times"], pred * np.max(score), label="prediction", alpha=0.7)
plt.xlabel("Time [s]")
plt.ylabel(score_name)
plt.title(f"Baseline VAD example: {example_id}")
plt.grid(True)
plt.legend()
plt.show()

display(pd.DataFrame([macro_f1_score(d["y"], pred)]))